In [34]:
import pandas as pd
df = pd.read_csv(r"C:\Users\zeid9\Documents\Scalable-MLOps-Pipeline-for-Credit-Risk-Prediction\data\credit_risk_dataset.csv")

In [35]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32581 entries, 0 to 32580
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   person_age                  32581 non-null  int64  
 1   person_income               32581 non-null  int64  
 2   person_home_ownership       32581 non-null  object 
 3   person_emp_length           31686 non-null  float64
 4   loan_intent                 32581 non-null  object 
 5   loan_grade                  32581 non-null  object 
 6   loan_amnt                   32581 non-null  int64  
 7   loan_int_rate               29465 non-null  float64
 8   loan_status                 32581 non-null  int64  
 9   loan_percent_income         32581 non-null  float64
 10  cb_person_default_on_file   32581 non-null  object 
 11  cb_person_cred_hist_length  32581 non-null  int64  
dtypes: float64(3), int64(5), object(4)
memory usage: 3.0+ MB


In [36]:
# Define a dictionary to map old column names to new, more intuitive names
columns_to_rename = {
    'person_age': 'age',                       # Rename to represent the borrower's age
    'person_home_ownership': 'home_status',    # Rename to indicate homeownership status
    'person_income': 'income',                 # Rename to represent the borrower's income level
    'person_emp_length': 'emp_years',          # Rename to indicate employment duration in years
    'loan_amnt': 'loan_amount',                # Rename to represent the loan amount
    'cb_person_cred_hist_length': 'credit_history',  # Rename to represent credit history length
    'cb_person_default_on_file': 'credit_default'  # Rename to indicate credit default status
}

# Apply the renaming to the dataset
df.rename(columns=columns_to_rename, inplace=True)

# Verify the updated column names
print("Updated column names:")
for column in df.columns:
    print(column)

Updated column names:
age
income
home_status
emp_years
loan_intent
loan_grade
loan_amount
loan_int_rate
loan_status
loan_percent_income
credit_default
credit_history


In [37]:
# Map 'Y' to 1 and 'N' to 0 in the 'credit_default' column
df['credit_default'] = df['credit_default'].map({'Y': 1, 'N': 0})

In [42]:
# Impute missing values for numerical columns using the median
print("Imputing missing values for numerical columns...")
for col in df.select_dtypes(include=['float64', 'int64']).columns:
    df[col] = df[col].fillna(df[col].median())

# Verify that missing values have been addressed
missing_counts_after = df.isnull().sum()

missing_counts_after


Imputing missing values for numerical columns...


age                    0
income                 0
home_status            0
emp_years              0
loan_intent            0
loan_grade             0
loan_amount            0
loan_int_rate          0
loan_status            0
loan_percent_income    0
credit_default         0
credit_history         0
dtype: int64

In [44]:
X = df.drop("credit_default", axis = 1)
y = df["credit_default"]

In [45]:
X.shape, y.shape

((32581, 11), (32581,))

In [46]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [48]:
X_train.shape, y_train.shape

((26064, 11), (26064,))

In [49]:
X_test.shape, y_test.shape

((6517, 11), (6517,))

In [50]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Define columns
categorical_cols = ['home_status', 'loan_intent', 'loan_grade']
numeric_cols = ['income', 'emp_years', 'loan_amount',
                'loan_int_rate', 'loan_status', 'loan_percent_income',
                'credit_history']

# Split raw data (before pipeline)
X = df[categorical_cols + numeric_cols]
y = df['credit_default']

# Train-test split
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Preprocessing
preprocessor = ColumnTransformer(transformers=[
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_cols),
    ('num', StandardScaler(), numeric_cols)
])

In [51]:
X_train_preprocessed = preprocessor.fit_transform(X_train)
X_test_preprocessed = preprocessor.transform(X_test)

In [54]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train_preprocessed, y_train)


In [56]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(max_features=10, n_estimators=1000, random_state=42)
rf.fit(X_train_res, y_train_res)


RandomForestClassifier(max_features=10, n_estimators=1000, random_state=42)

In [58]:
# Evaluate
y_pred = rf.predict(X_test_preprocessed)

In [67]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
# Accuracy
print("Accuracy Score:", accuracy_score(y_test, y_pred))

# Classification report
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# Confusion matrix
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# ROC AUC Score
print("ROC AUC Score:", roc_auc_score(y_test, y_pred))


Accuracy Score: 0.8282952278655823

Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.85      0.89      5368
           1       0.51      0.75      0.61      1149

    accuracy                           0.83      6517
   macro avg       0.72      0.80      0.75      6517
weighted avg       0.86      0.83      0.84      6517


Confusion Matrix:
 [[4541  827]
 [ 292  857]]
ROC AUC Score: 0.7959024337887285


In [53]:
# Pipeline
rf_model_pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", RandomForestClassifier(max_features= 10, n_estimators=1000))
])

rf_model_pipeline.fit(X_train,y_train)


KeyboardInterrupt: 

In [ ]:
rf_model_pipeline.score(X_test, y_test)

In [ ]:
y_pred = rf_model_pipeline.predict(X_test)

In [ ]:
accuracy_score(y_pred, y_test)

In [ ]:
y_pred, y_test

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
import numpy as np

# Define your pipeline
rf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Define hyperparameter grid
param_grid = {
    'classifier__max_features': np.arange(1, 6, 1),
    'classifier__n_estimators': np.arange(10, 110, 10)
}

# Grid Search
grid = GridSearchCV(
    estimator=rf_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='recall,  # prioritize recall due to imbalanced data
    n_jobs=-1,
    verbose=2
)

# Fit on the original X_train and y_train (not resampled)
grid.fit(X_train, y_train)


Fitting 5 folds for each of 50 candidates, totalling 250 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(transformers=[('cat',
                                                                         OneHotEncoder(drop='first',
                                                                                       handle_unknown='ignore'),
                                                                         ['home_status',
                                                                          'loan_intent',
                                                                          'loan_grade']),
                                                                        ('num',
                                                                         StandardScaler(),
                                                                         ['income',
                                                                          'emp_years',
                                                                          'loan_amount',
                                                                          'loan_int_rate',
                                                                          'loan_status',
                                                                          'loan_percent_income',
                                                                          'credit_history'])])),
                                       ('smote', SMOTE(random_state=42)),
                                       ('classifier',
                                        RandomForestClassifier(random_state=42))]),
             n_jobs=-1,
             param_grid={'classifier__max_features': array([1, 2, 3, 4, 5]),
                         'classifier__n_estimators': array([ 10,  20,  30,  40,  50,  60,  70,  80,  90, 100])},
             scoring='precision', verbose=2)

In [87]:
print("The best parameters are %s with a score of %0.2f"
      % (grid.best_params_, grid.best_score_))

The best parameters are {'classifier__max_features': np.int64(5), 'classifier__n_estimators': np.int64(10)} with a score of 0.51


In [88]:

import pandas as pd

grid_results = pd.concat([pd.DataFrame(grid.cv_results_["params"]),pd.DataFrame(grid.cv_results_["mean_test_score"], columns=["Recall"])],axis=1)
grid_results

,classifier__max_features,classifier__n_estimators,Recall
0,1,10,0.504670
1,1,20,0.506804
2,1,30,0.509122
3,1,40,0.509444
4,1,50,0.510053
5,1,60,0.510170
6,1,70,0.508165
7,1,80,0.507778
8,1,90,0.509816
9,1,100,0.509670


In [89]:
grid_contour = grid_results.groupby(
    ['classifier__max_features', 'classifier__n_estimators']
).mean()

grid_contour_expanded = grid_contour.unstack(level='classifier__n_estimators')
grid_contour_expanded

Recall                                          \
classifier__n_estimators       10        20        30        40        50    
classifier__max_features                                                     
1                         0.504670  0.506804  0.509122  0.509444  0.510053   
2                         0.505232  0.506920  0.509902  0.507346  0.507762   
3                         0.510929  0.508419  0.513799  0.512760  0.510587   
4                         0.512446  0.508263  0.507655  0.510861  0.509814   
5                         0.514738  0.509841  0.508931  0.508441  0.511876   

                                                                            
classifier__n_estimators       60        70        80        90        100  
classifier__max_features                                                    
1                         0.510170  0.508165  0.507778  0.509816  0.509670  
2                         0.507150  0.507623  0.508987  0.509945  0.507750  
3                         0.509696  0.508470  0.510572  0.509405  0.510765  
4                         0.512537  0.510665  0.509010  0.510258  0.510688  
5                         0.509272  0.508095  0.508418  0.509255  0.508195

In [90]:

x = grid_contour_expanded.columns.levels[1].values
y = grid_contour_expanded.index.values
z = grid_contour_expanded.values

In [91]:

import plotly.graph_objects as go

# x = n_estimators (columns)
# y = max_features (rows)
# z = Recall

fig = go.Figure(data=[go.Surface(z=z, x=x, y=y)])

fig.update_layout(
    title='Hyperparameter Tuning: Accuracy by n_estimators & max_features',
    scene=dict(
        xaxis=dict(title='n_estimators'),
        yaxis=dict(title='max_features'),
        zaxis=dict(title='Recall'),
    ),
    autosize=False,
    width=800,
    height=800,
    margin=dict(l=65, r=50, b=65, t=90)
)

fig.show()


In [95]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# Final pipeline
final_rf_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(
        max_features=4,
        n_estimators=100,
        random_state=42
    ))
])

# Train final model on the full training set
final_rf_pipeline.fit(X_train, y_train)

# Predict on test set
y_pred = final_rf_pipeline.predict(X_test)

# Evaluate
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

print("Accuracy Score:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy Score: 0.8315175694337885

Classification Report:
               precision    recall  f1-score   support

           0       0.95      0.84      0.89      5368
           1       0.51      0.79      0.62      1149

    accuracy                           0.83      6517
   macro avg       0.73      0.81      0.76      6517
weighted avg       0.87      0.83      0.84      6517


Confusion Matrix:
 [[4515  853]
 [ 245  904]]
